# pyda Repository — Long Time-Series Storage and Retrieval

Demonstrates the recommended pattern for long continuous data streams in LTPDA repositories:

1. **Store** — split a 24-hour signal into 1-hour segments, set `t0` on each, submit all.
2. **Retrieve by time range** — `repo.get('channel', t0, t1)` retrieves overlapping segments,
   concatenates, and crops to the exact window.
3. **Edge cases** — queries spanning one segment, queries between segments.

This mirrors the MATLAB toolbox workflow: `getObjectIdInTimespan()` + `retrieve()` + stitch.

---

⚠️ **Do not save this notebook with real credentials filled in.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from pyda.tsdata import TSData
from pyda.repo import LTPDARepository

## Connect

In [ ]:
import os
os.environ.setdefault('LTPDA_HOST', 'localhost')
os.environ.setdefault('LTPDA_PORT', '3306')
os.environ.setdefault('LTPDA_DB',   'ltpda_repo')
os.environ.setdefault('LTPDA_USER', 'alice')
os.environ.setdefault('LTPDA_PASS', 'mysql_secret')

repo = LTPDARepository.from_env()
repo.connect()
print('Connected.')

## 1. Store a 24-hour signal as 1-hour segments

Long continuous recordings are stored as shorter segments — the repository design intentionally
avoids single very large blobs. Each segment is a standalone `TSData` object with an absolute
`t0` that marks its position in the timeline.

In [ ]:
CHANNEL   = 'SEISM_X'
FS        = 10          # Hz
SEG_SECS  = 3600        # 1 hour per segment
N_SEGS    = 24          # 24 segments = 24 hours
DAY_START = datetime(2024, 1, 15, 0, 0, 0)

segment_ids = []

for i in range(N_SEGS):
    seg_t0 = DAY_START + timedelta(seconds=i * SEG_SECS)

    seg = TSData.randn(nsecs=SEG_SECS, fs=FS, name=CHANNEL, yunits='m/s^2')
    seg.t0 = seg_t0
    seg.description = f'Hour {i:02d}:00 – {i+1:02d}:00 UTC'

    result = repo.submit(
        seg,
        experiment_title='24-hour seismometer run 2024-01-15',
        experiment_desc='Continuous seismometer X-axis recording for full day 2024-01-15',
        analysis_desc=f'Raw segment hour {i:02d} — no processing applied',
        quantity='seismic acceleration',
        keywords=f'seismometer, SEISM_X, hour{i:02d}',
    )
    segment_ids.append(result.id)
    print(f'  Hour {i:02d}: id={result.id}, t0={seg_t0}')

print(f'\nStored {N_SEGS} segments, ids {segment_ids[0]}–{segment_ids[-1]}')

## 2. Retrieve a 6-hour window spanning multiple segments

`repo.get()` finds all segments whose stored time span overlaps `[t0, t1]`,
retrieves each one, sorts by `t0`, concatenates, and crops to the exact requested window.
The returned object is a single `TSData`.

In [ ]:
t0_query = datetime(2024, 1, 15, 2, 0, 0)   # 02:00 UTC
t1_query = datetime(2024, 1, 15, 8, 0, 0)   # 08:00 UTC  (6 hours)

ts_window = repo.get(CHANNEL, t0=t0_query, t1=t1_query)

print(ts_window)
print(f'\nRetrieved window:')
print(f'  t0     = {ts_window.t0}')
print(f'  nsecs  = {ts_window.nsecs():.1f} s  ({ts_window.nsecs()/3600:.1f} h)')
print(f'  npts   = {len(ts_window.yaxis.data)}')
print(f'  fs     = {ts_window.fs()} Hz')

## 3. Visualize

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

t_hours = ts_window.xaxis.data / 3600   # relative seconds → hours from t0
ax.plot(t_hours, ts_window.yaxis.data, lw=0.5, color='steelblue')

ax.set_xlabel('Time from query t0  [h]')
ax.set_ylabel(f'{ts_window.name}  [{ts_window.yaxis.units}]')
ax.set_title(f'{ts_window.name}: {t0_query} – {t1_query} UTC')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Edge case — query spanning exactly one segment

A query that lies entirely within one stored segment still works correctly.

In [ ]:
t0_single = datetime(2024, 1, 15, 5, 15, 0)   # 05:15 UTC
t1_single = datetime(2024, 1, 15, 5, 45, 0)   # 05:45 UTC  (30 min within hour 05)

ts_single = repo.get(CHANNEL, t0=t0_single, t1=t1_single)

print(f'nsecs  = {ts_single.nsecs():.1f} s  (expected ~{(t1_single - t0_single).total_seconds():.0f} s)')
print(f'npts   = {len(ts_single.yaxis.data)}  (expected ~{int((t1_single - t0_single).total_seconds() * FS)})')

## 5. Edge case — query that falls between stored segments

If no segments overlap the requested window, `repo.get()` returns `None`.

In [ ]:
# Query a channel that does not exist in the repository
result_none = repo.get('NONEXISTENT_CHANNEL',
                        t0=datetime(2024, 1, 15, 0, 0),
                        t1=datetime(2024, 1, 15, 1, 0))

print(f'Result for non-existent channel: {result_none}')   # should print None

## 6. Check for the most recent segment (`get_latest`)

Returns the `ObjectMeta` for the stored segment with the most recent end time.

In [ ]:
latest = repo.get_latest(CHANNEL)
if latest is not None:
    print(f'Latest segment for {CHANNEL!r}:')
    print(f'  id    = {latest.id}')
    print(f'  t0    = {latest.t0}')
    print(f'  nsecs = {latest.nsecs}')
    end_time = latest.t0 + timedelta(seconds=latest.nsecs) if latest.t0 and latest.nsecs else None
    print(f'  end   = {end_time}')
else:
    print(f'No segments found for {CHANNEL!r}')

## 7. Search with timespan overlap filter

`repo.find(timespan=(t0, t1))` uses the MATLAB-compatible XPath query on the keywords XML
field, so it finds segments whose stored timespan overlaps the requested window —
without downloading any binary data.

In [ ]:
overlapping = repo.find(
    name=CHANNEL,
    timespan=(datetime(2024, 1, 15, 10, 0), datetime(2024, 1, 15, 14, 0)),
)

print(f'Segments overlapping 10:00–14:00 UTC:')
for r in overlapping:
    print(f'  id={r.id}  t_start={r.t_start}  t_stop={r.t_stop}')

## 8. Disconnect

In [ ]:
repo.close()
print('Connection closed.')